# Data Cleaning

The cleaning process took care of:  
- removing duplicates  
- removing ads with empty descriptions  
- removing html tags  
- applying regex to mask phone numbers, emails and names  

Completely remove all the personal information from the dataset in a reproducible way would be a project on itself. Since my focus is on skill retrieval, I will not commit the raw/cleaned data to the repo.

In [1]:
import json
import pandas as pd
import re, html

In [2]:
with open("../data/raw_active_ads_2511.json", encoding="utf-8") as f:
    raw_active = pd.DataFrame(json.load(f))

raw_active_df = raw_active.copy()

with open("../data/raw_historic_ads_2511.json", encoding="utf-8") as f:
    raw_historic= pd.DataFrame(json.load(f))

raw_historic_df = raw_historic.copy()

print(f"Read dataframes:\n"
      f"Active ads: {len(raw_active_df)}\n"
      f"Historic ads: {len(raw_historic_df)}")

Read dataframes:
Active ads: 289
Historic ads: 2644


In [3]:
raw_historic_df = raw_historic_df.drop_duplicates(subset="version_id", keep="first") # removing duplicates based on version_id just in case
raw_historic_df = raw_historic_df.drop(columns=["version_id", "deadline"]) # dropping unused cols
raw_historic_df.head()

,id,title,employer,municipality,region,occupation,published,description
0,30321971,IT-konsult,Capgemini Sverige AB,NaN,NaN,"Affärskonsult, IT",2026-01-01T00:00:00,Om jobbet\nCapgemini strävar efter att vara en...
1,30321996,Application Consultant,Capgemini Sverige AB,NaN,NaN,"Affärskonsult, IT",2026-01-01T00:00:00,"Together with a strong, multinational Capgemin..."
2,30321974,Sales Representative,Capgemini Sverige AB,NaN,NaN,"Affärskonsult, IT",2026-01-01T00:00:00,Om jobbet \nCapgemini strävar efter att vara e...
3,30412310,Dataanalytiker/Data Analyst,UTBETALNINGSMYNDIGHETEN,Stockholm,Stockholms län,"Metodanalytiker, IT",2026-01-02T00:00:00,Vill du bidra till att lösa en av vår tids stö...
4,30430148,Chief Architect till MaintMaster!,A Hub AB,Linköping,Östergötlands län,IT-arkitekt/Lösningsarkitekt,2026-01-02T08:21:34,Vill du arbeta i ett internationellt växande p...


In [4]:
raw_active_df = raw_active_df.drop(columns="deadline")
raw_active_df.head()

,id,title,employer,municipality,region,occupation,published,description
0,30523189,PLM Teamcenter,ITMOTIVE in Sweden Aktiebolag,NaN,NaN,"Verksamhetskonsult, IT",2026-01-27T01:04:50,"PLM Teamcenter Developer, Solution Architect.\..."
1,30896666,Teamchef Data & Analytics till Dizparc,Bravura Sverige AB,Linköping,Östergötlands län,"Affärskonsult, IT",2026-04-13T14:45:16,Dizparc i Linköping bygger sitt nästa affärsbe...
2,30896809,Senior BI-konsult – forma framtidens datadrivn...,A Hub AB,Karlstad,Värmlands län,"Verksamhetskonsult, IT",2026-04-13T14:58:59,Vill du arbeta i en roll där du tar fullt ansv...
3,30903986,Till vår kund Invid söker vi Affärssystems kon...,FollowU AB,Jönköping,Jönköpings län,"Verksamhetskonsult, IT",2026-04-14T15:38:38,Business Central-konsulter till INVID\nVill du...
4,30915541,IT-Specialist Windows Server,Techrytera AB,Stockholm,Stockholms län,IT-arkitekt/Lösningsarkitekt,2026-04-16T14:50:43,Om oss På Techrytera AB börjar rekryteringen m...


In [5]:
# concat the dfs
raw_df_combined = pd.concat([raw_active_df, raw_historic_df], axis=0, ignore_index=True) 

combined_length = len(raw_df_combined)

print(f"Active ads ({len(raw_active_df)}) + Historic ads ({len(raw_historic_df)}) = {len(raw_active_df) + len(raw_historic_df)}")
print(f"Combined df: {combined_length}")
raw_df_combined.info()

Active ads (289) + Historic ads (2644) = 2933
Combined df: 2933
<class 'pandas.DataFrame'>
RangeIndex: 2933 entries, 0 to 2932
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            2933 non-null   str  
 1   title         2933 non-null   str  
 2   employer      2933 non-null   str  
 3   municipality  2758 non-null   str  
 4   region        2758 non-null   str  
 5   occupation    2933 non-null   str  
 6   published     2933 non-null   str  
 7   description   2933 non-null   str  
dtypes: str(8)
memory usage: 183.4 KB


In [6]:
raw_df_combined.head()

,id,title,employer,municipality,region,occupation,published,description
0,30523189,PLM Teamcenter,ITMOTIVE in Sweden Aktiebolag,NaN,NaN,"Verksamhetskonsult, IT",2026-01-27T01:04:50,"PLM Teamcenter Developer, Solution Architect.\..."
1,30896666,Teamchef Data & Analytics till Dizparc,Bravura Sverige AB,Linköping,Östergötlands län,"Affärskonsult, IT",2026-04-13T14:45:16,Dizparc i Linköping bygger sitt nästa affärsbe...
2,30896809,Senior BI-konsult – forma framtidens datadrivn...,A Hub AB,Karlstad,Värmlands län,"Verksamhetskonsult, IT",2026-04-13T14:58:59,Vill du arbeta i en roll där du tar fullt ansv...
3,30903986,Till vår kund Invid söker vi Affärssystems kon...,FollowU AB,Jönköping,Jönköpings län,"Verksamhetskonsult, IT",2026-04-14T15:38:38,Business Central-konsulter till INVID\nVill du...
4,30915541,IT-Specialist Windows Server,Techrytera AB,Stockholm,Stockholms län,IT-arkitekt/Lösningsarkitekt,2026-04-16T14:50:43,Om oss På Techrytera AB börjar rekryteringen m...


In [7]:
# remove duplicates
clean_df_1 = raw_df_combined.copy()
clean_df_1 = clean_df_1.drop_duplicates(subset="id", keep="first")
clean_df_1 = clean_df_1.drop_duplicates(subset=["title", "employer", "description"], keep="first")

clean_1_length = len(clean_df_1)

print(f"Original length: {combined_length} rows.\n"
      f"{combined_length - clean_1_length} duplicates found and removed.\n"
      f"Length after the first cleaning step: {clean_1_length} rows.")

Original length: 2933 rows.
199 duplicates found and removed.
Length after the first cleaning step: 2734 rows.


In [8]:
# remove empty descriptions
clean_df_2 = clean_df_1[clean_df_1["description"].fillna("").str.strip().ne("")]

clean_2_length = len(clean_df_2)

print(f"Removed {clean_1_length - clean_2_length} rows with empty descriptions.\n"
      f"Length after the second cleaning step: {clean_2_length} rows.")

Removed 5 rows with empty descriptions.
Length after the second cleaning step: 2729 rows.


In [9]:
patterns = {
    "phone": r"(?<!\d)(?<!\d )(?:\+46|0046|0)[\s-]?(?:\d[\s-]?){7,10}(?!\d)",
    "email": r"[\w.%+-]+@[\w.-]+\.[A-Za-z]{2,}",
    "html": r"<[^>]+>"
}

ignore = r"\d{2}-\d{2}-\d{4}|\d{5}-\d{4}" # ignores dates

clean_df_3 = clean_df_2.copy()

In [10]:
# Mask phones
clean_df_3["description_original"] = clean_df_3["description"]

def mask_phone(match):
    text = match.group()
    if re.fullmatch(ignore, text.strip()):   # date or case number -> keep as is
        return text
    return "[phone]"

clean_df_3["description"] = clean_df_3["description_original"].fillna("").apply(
    lambda t: re.sub(patterns["phone"], mask_phone, t)
)

masked_phone = clean_df_3[clean_df_3["description"].str.contains(r"\[phone\]", regex=True)]
print("Masked ads:", len(masked_phone))

# for _, row in masked_phone.head(5).iterrows():
#     print("-" * 50)
#     print("ID:", row["id"])
#     print("Masked from:", re.findall(patterns["phone"], row["description_original"]))
#     pos = row["description"].find("[phone]")
#     print("After:", row["description"][max(0, pos - 80): pos + 80])

Masked ads: 39


In [11]:
# Mask emails
clean_df_3["description"] = clean_df_3["description"].fillna("").str.replace(
    patterns["email"], "[email]", regex=True)

masked_email = clean_df_3[clean_df_3["description"].str.contains(r"\[email\]", regex=True)]
print("Ads with masked emails:", len(masked_email))

# for _, row in masked_email.head(5).iterrows():
#     print("-" * 50)
#     print("ID:", row["id"])
#     print("Masked from:", re.findall(patterns["email"], row["description_original"]))
#     pos = row["description"].find("[email]")
#     print("After:", row["description"][max(0, pos - 80): pos + 80])

Ads with masked emails: 73


In [12]:
# remove HTML tags
clean_df_3["description"] = (
    clean_df_3["description"].fillna("")
    .str.replace(patterns["html"], " ", regex=True) # tags -> space, so words don't merge
    .apply(html.unescape) # &amp; -> &, &auml; -> ä, &nbsp; -> space
    .str.replace(r"\s+", " ", regex=True) # collapse repeated spaces/newlines
    .str.strip()
)

masked_html = clean_df_3[clean_df_3["description_original"].str.contains(pat=r"<[^>]+>", regex=True)]
print("Ads with removed html:", len(masked_html))

Ads with removed html: 3


### Finding personal names

In [13]:
cue = r"\b(?:kontakta|kontaktperson(?:\s+för frågor)?|rekryterande chef|ansvarig rekryterare|rekryteringsansvarig|rekryterare|hör av dig till)\b"
name = r"[A-ZÅÄÖ][a-zåäö]+(?:[-'][A-ZÅÄÖ]?[a-zåäö]+)?(?:\s+[A-ZÅÄÖ][a-zåäö]+(?:[-'][A-ZÅÄÖ]?[a-zåäö]+)?){0,2}"
name_pattern = re.compile(rf"({cue}\s*[:\-]?\s*(?:hos\s+)?){name}", re.IGNORECASE)

clean_df_4 = clean_df_3.copy()
clean_df_4["description"] = clean_df_4["description"].str.replace(name_pattern, r"\1[name]", regex=True)

In [14]:
masked_name = clean_df_4[clean_df_4["description"].str.contains(r"\[name\]", regex=True)]
print("Ads with masked name:", len(masked_name))

# for _, row in masked_name.head(50).iterrows():
#     print("-" * 50)
#     print("ID:", row["id"])
#     pos = row["description"].find("[name]")
#     print("After:", row["description"][max(0, pos - 80): pos + 80])

Ads with masked name: 346


In [15]:
clean_df = clean_df_4.copy()
clean_df = clean_df.drop(columns="description_original")
clean_df.head()

,id,title,employer,municipality,region,occupation,published,description
0,30523189,PLM Teamcenter,ITMOTIVE in Sweden Aktiebolag,NaN,NaN,"Verksamhetskonsult, IT",2026-01-27T01:04:50,"PLM Teamcenter Developer, Solution Architect. ..."
1,30896666,Teamchef Data & Analytics till Dizparc,Bravura Sverige AB,Linköping,Östergötlands län,"Affärskonsult, IT",2026-04-13T14:45:16,Dizparc i Linköping bygger sitt nästa affärsbe...
2,30896809,Senior BI-konsult – forma framtidens datadrivn...,A Hub AB,Karlstad,Värmlands län,"Verksamhetskonsult, IT",2026-04-13T14:58:59,Vill du arbeta i en roll där du tar fullt ansv...
3,30903986,Till vår kund Invid söker vi Affärssystems kon...,FollowU AB,Jönköping,Jönköpings län,"Verksamhetskonsult, IT",2026-04-14T15:38:38,Business Central-konsulter till INVID Vill du ...
4,30915541,IT-Specialist Windows Server,Techrytera AB,Stockholm,Stockholms län,IT-arkitekt/Lösningsarkitekt,2026-04-16T14:50:43,Om oss På Techrytera AB börjar rekryteringen m...


In [16]:
# Saving
with open("../data/clean_ads.json", "w", encoding="utf-8") as f:
    json.dump(clean_df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)